# 04 · Cómo se mide y contra qué se compara

Antes de comparar modelos hay que acordar cuatro cosas: con qué datos se decide y con cuáles se confirma, qué referencias tiene que superar cualquier modelo, cómo se estima el modelo base (la que resultó más importante de lo esperado) y cómo se asegura que no se use información del futuro.

## Protocolo

- **Cuándo se pronostica:** el pronóstico del día *d* se hace hacia el mediodía de *d−1*. En ese momento se conoce el MDA hasta *d* y el MTR solo hasta *d−8*. El modelo de cada día se entrena únicamente con días ≤ *d−8*.
- **Desarrollo (1 de julio al 31 de diciembre de 2024, 184 días):** aquí se toman todas las decisiones: qué variables, qué configuración, qué combinaciones.
- **Test (1 de enero al 30 de junio de 2025, 181 días):** se abre al final, en el notebook 11, con todo ya decidido. La escalera del notebook 08 también lo muestra, solo como contraste: ninguna decisión se toma mirándolo. Si se eligiera mirando el test, el resultado reportado sería optimista: mediría la suerte al elegir, no la calidad del modelo.
- **Entrenamiento:** desde el 5 de febrero de 2024 (ver notebook 03).
- **Cómo se mide el error:** con el error absoluto medio (MAE), en MXN/MWh: el promedio, sobre las 24 horas de los días evaluados, de |MTR − pronóstico|. Para comparar modelos se usa el cambio porcentual del MAE contra el MDA, (MAE del modelo − MAE del MDA) / MAE del MDA, donde negativo es mejor. Cada tabla da los dos números. También se reporta el MAE sin los días de pico (algún MTR mayor a 3,000), porque unos pocos picos dominan el promedio.
- **Cómo se sabe si una diferencia es real:** con 184 días, una mejora de 1% puede ser suerte. Se usa la prueba de Giacomini–White, que compara el error absoluto diario de dos modelos y además pregunta si la diferencia de ayer anticipa la de hoy: una ventaja que va y viene cuenta menos que una persistente. Su *p* es la probabilidad de ver una diferencia así de consistente si los dos modelos fueran igual de buenos; menor a 0.05 se lee como "no es un par de días con suerte". Junto con la *p* se reporta en cuántos de los 6 meses gana el modelo. Cada prueba aislada puede salir bien por azar, así que en desarrollo la *p* es solo una guía, y en el test se usa una única vez, para el modelo final.
- **Cómo se elige entre versiones, mezclas y modelo final:** con el peor mes, es decir, el mayor cambio mensual del error contra el MDA en los seis meses de desarrollo (menos es mejor). En los primeros cálculos del proyecto se usó el error medio de todo el desarrollo. Al comparar criterios en el notebook 11 se vio que el error medio lo mueven los meses extremos, algo natural en datos con cambios abruptos de un mes a otro, y que el peor mes anticipa mejor qué modelo se sostiene en el test; por eso se volvió a correr toda la elección (versiones de la escalera, mezclas y modelo final) con el peor mes. Los pronósticos de cada modelo no cambian, porque no dependen de este criterio. En cambio, los hiperparámetros dentro de cada modelo (alpha de LEAR, tamaño de árbol del GB, configuración de las redes) se escogieron con el error medio de desarrollo, que era la suposición de ese momento; lo consistente sería volver a escogerlos con el peor mes y reentrenar los modelos, algo que queda pendiente. Una diferencia de menos de 0.5 puntos no se distingue del ruido y se prefiere la versión más sencilla.

In [1]:
import os
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'): os.environ[k] = '1'   # evita sobresuscripción en procesos paralelos
import sys, warnings, time, pickle
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dataclasses import replace
from pml import plotstyle as ps
ps.aplicar()
FIG = ROOT / 'reports/figures'; REP = ROOT / 'reports'

# Contexto de datos y periodo de desarrollo (jul–dic 2024): aquí se decide todo
from pml import features2 as f2, exp
from pml.metrics import mae
c0 = f2.cargar_ctx(); DEV = exp.idx_dev(); Y, X = c0.Y, c0.X
f = lambda d: d.loc[DEV].values
# Las dos referencias contra las que se mide todo: usar el MDA tal cual y repetir el MTR de hace 14 días
refs = {'MDA': mae(f(Y), f(X)), 'naive d-14': mae(f(Y), f(Y.shift(14)))}
print('desarrollo:', DEV.min().date(), '→', DEV.max().date(), f'({len(DEV)} días)')

desarrollo: 2024-07-01 → 2024-12-31 (184 días)


## 1. Referencias

Antes de construir cualquier modelo, conviene saber qué tan difícil es ganarle a reglas sencillas:

- **Repetir el MTR de hace dos semanas** (*naive d−14*), el mismo día de la semana. Es la referencia más tonta, y se usa para expresar los errores en proporción (rMAE).
- **Repetir el MTR más reciente que existe** (*naive d−8*).
- **El MDA tal cual.** Es la referencia que importa: si un modelo no le gana al MDA, en la práctica no sirve.
- **El MDA corregido por lo que se equivocó la semana pasada:** el MDA más la mediana, hora por hora, de la diferencia MTR − MDA entre *d−14* y *d−8*.
- **El modelo nulo:** el MDA más la diferencia típica de toda la historia conocida. Es lo que haría un modelo sin ninguna variable. Si un modelo no le gana al nulo, sus variables no aportan.


In [2]:
# Referencias sin modelo: cada una es una regla fija que se puede calcular con lo que se sabe en d−1
Yt, Xt = f2._t(c0); sp = Y - X
# el error mediano del MDA en los días d−8 a d−14 (los últimos ya publicados), hora por hora
med_rec = pd.DataFrame(np.median(np.stack([sp.shift(k).values for k in range(8, 15)]), axis=0), index=sp.index, columns=sp.columns)
# "nulo": el MDA más la mediana de todo su error histórico conocido (un modelo que sólo aprende un sesgo)
def nulo(c):
    Yt_, Xt_ = f2._t(c); s = Yt_ - Xt_
    m = np.array([np.median(s.loc[f2.TRAIN_START:d - pd.Timedelta(days=8)].values) for d in DEV])
    return pd.DataFrame(f2.inv(Xt_.loc[DEV].values + m[:, None], c), index=DEV, columns=X.columns)
REF = {'MDA': X, 'naive d-14': Y.shift(14), 'naive d-8': Y.shift(8), 'MDA + spread mediano d−8…d−14': X + med_rec, 'nulo (MDA + mediana del spread)': nulo(c0)}
T = exp.tabla_dev(c0, REF, 'MDA', [k for k in REF if k != 'MDA'], refs)
# rMAE: error relativo al naive de 14 días (menor que 1 = mejor que repetir el pasado)
T.insert(1, 'rMAE', T.MAE / refs['naive d-14'])
T = exp.legible(T.drop(columns='dMAE_pct')); display(T); T.to_csv(REP / '04_referencias_dev.csv')

,MAE,rMAE,MAE sin días de pico,vs MDA (%),p,meses mejor,de
MDA,221.62,0.64,168.33,0.00,NaN,<NA>,<NA>
naive d-14,347.53,1.00,310.85,56.81,0.000,0,6
naive d-8,339.68,0.98,299.95,53.27,0.000,0,6
MDA + spread mediano d−8…d−14,249.38,0.72,203.32,12.53,0.000,0,6
nulo (MDA + mediana del spread),222.38,0.64,169.58,0.34,0.006,1,6


Repetir el MTR pasado se equivoca 53–57% más que el MDA, y corregir el MDA con lo que pasó la semana anterior lo empeora 12.5%: la sorpresa de una semana no se repite la siguiente. El nulo queda prácticamente empatado con el MDA. **La vara está alta:** el MDA ya es un pronóstico muy bueno, y las correcciones ingenuas no lo mejoran.

## 2. El modelo base, y dónde estuvo la primera sorpresa

El modelo base es **LEAR**, el modelo de referencia de la literatura de precios eléctricos (Lago et al. 2021). En palabras simples: para cada una de las 24 horas, una regresión lineal que aprende cuánto corregir al MDA a partir del MTR y el MDA de días anteriores. Tiene una penalización que apaga las variables que no ayudan (LASSO), así que con pocos datos se queda con lo esencial. Se reentrena cada día con tres ventanas de historia (8 semanas, 12 semanas y toda), y se promedian los tres pronósticos.

Se pronostica la **corrección** (MTR − MDA) y se suma al MDA. Las variables son el MTR de *d−8, d−9, d−10* y *d−14*, el MDA de *d, d−1* y *d−7*, y lo que se equivocó el MDA en *d−8* y *d−14*.

**La primera versión salió 5.6% peor que el MDA**, peor incluso que el nulo. Al buscar la causa, se encontró el problema en el punto de partida de la regresión (el intercepto). La regresión arranca del *promedio* de la corrección, y ese promedio está inflado por unos pocos picos de 2024. Pero el error que se mide (MAE) se minimiza con la *mediana*, no con el promedio. El modelo arrancaba con un sesgo hacia arriba que ninguna variable corregía. La solución fue ajustar los coeficientes igual y fijar el punto de partida en la mediana.

La tabla muestra esa historia junto con otras dos decisiones: en qué escala se miden los precios, y si se recortan los valores extremos al entrenar.


In [3]:
# El mismo LEAR (rezagos de MTR y MDA) con distintas formas de estimarlo. Lo único que cambia es:
# - el intercepto: media (mínimos cuadrados) o mediana (más acorde con una métrica de error absoluto);
# - la transformación del precio: lineal o asinh con distintas escalas (×1, ×0.5, …: qué tanto se "aplastan" las colas);
# - recortar el 1% más extremo del objetivo al entrenar.
cb = replace(c0, escala=0.5)
CF = {
 'intercepto media · asinh ×1':         dict(nombre='B0 asinh', c=c0, centro='media'),
 'intercepto media · lineal (sin asinh)': dict(nombre='T1 lineal', c=replace(c0, modo='lineal'), centro='media'),
 'intercepto mediana · asinh ×1':       dict(nombre='Bm asinh x1', c=c0, centro='mediana'),
 'intercepto mediana · asinh ×0.5':     dict(nombre='Bm asinh x0.5', c=cb, centro='mediana'),
 'intercepto mediana · asinh ×0.25':    dict(nombre='Bm asinh x0.25', c=replace(c0, escala=0.25), centro='mediana'),
 'intercepto mediana · asinh ×0.1':     dict(nombre='Bm asinh x0.1', c=replace(c0, escala=0.1), centro='mediana'),
 'intercepto mediana · asinh ×0.5 · recorte 1%': dict(nombre='Bm x0.5 winsor 1%', c=cb, centro='mediana', w=0.01),
}
FC = {'MDA': X, 'nulo (MDA + mediana del spread)': REF['nulo (MDA + mediana del spread)']}
# cada configuración se entrena (o se lee de caché) con la misma rutina: recalibración diaria, tres ventanas
for k, cfg in CF.items():
    FC[k], _ = exp.correr_y_evaluar(cfg['c'], cfg['nombre'], ['G0', 'G2'], winsor=cfg.get('w'), centro=cfg['centro'])
T = exp.legible(exp.tabla_dev(c0, FC, 'MDA', [k for k in FC if k != 'MDA'], refs).drop(columns='dMAE_pct')); display(T); T.to_csv(REP / '04_estimador_dev.csv')

,MAE,MAE sin días de pico,vs MDA (%),p,meses mejor,de
MDA,221.62,168.33,0.00,NaN,<NA>,<NA>
nulo (MDA + mediana del spread),222.38,169.58,0.34,0.006,1,6
intercepto media · asinh ×1,233.93,183.50,5.56,0.000,0,6
intercepto media · lineal (sin asinh),264.87,223.79,19.51,0.000,0,6
intercepto mediana · asinh ×1,219.83,166.95,-0.81,0.254,4,6
intercepto mediana · asinh ×0.5,220.09,168.24,-0.69,0.266,5,6
intercepto mediana · asinh ×0.25,221.25,170.31,-0.17,0.459,3,6
intercepto mediana · asinh ×0.1,316.73,278.92,42.92,0.355,0,6
intercepto mediana · asinh ×0.5 · recorte 1%,219.22,166.61,-1.08,0.237,5,6


Lo que muestra la tabla:

- **Partir de la mediana** convierte una base 5.6% peor que el MDA en una ligeramente mejor (−0.8%). Una línea de código movió más que cualquier variable que se probó después. La lección: el modelo tiene que perseguir lo mismo que la métrica.
- **La escala de los precios importa.** Trabajar con los precios tal cual es 20% peor que el MDA. Con picos de 13,000 sobre una mediana de 500, unos pocos días dominan el ajuste. Se usa una escala que comprime los picos (asinh: se comporta como lineal cerca de los valores normales y como logaritmo en los extremos, y a diferencia del logaritmo admite precios negativos).
- **Cuánto se comprime** casi no cambia el resultado entre ×1 y ×0.25. Con ×0.1 el modelo se rompe, porque al deshacer la compresión cualquier error se amplifica.
- **Recortar el 1% de los valores más extremos al entrenar** (el pronóstico no se toca) da el mejor error de desarrollo, 219.2. La mejora sobre el MDA es de ~1%, y todavía no se distingue del azar (p ≈ 0.24).

**Decisión:** la base para todo lo que sigue es LEAR, partiendo de la mediana, con la escala comprimida (×0.5) y el recorte de 1%. Es ligeramente mejor que el MDA en desarrollo, sin que esté demostrado. Los notebooks 05 a 10 intentan mejorarla.

## 3. Que ninguna variable use información del futuro

Un modelo que viera el futuro, aunque fuera por un descuido, parecería excelente en desarrollo y fallaría en la práctica. Por eso, cada variable pasa por una prueba mecánica: para varios días se reemplaza por números al azar todo lo que no estaría disponible al emitir el pronóstico, y se verifica que la fila de variables de ese día no cambie. La prueba también tiene un control positivo: una variable construida a propósito con el MTR de *d−7* (aún no publicado) debe ser detectada, y lo es (`tests/test_features2.py`).


In [4]:
# Prueba de no fuga por grupo de variables: para varios días se sustituye por basura todo lo que no estaría disponible al emitir el pronóstico
# (MTR posterior a d−8, MDA posterior a d, y las demás fuentes posteriores a su hora de publicación) y se cuenta cuántas columnas cambian
from pml.todo import TODO
c = f2.con_nodos(c0)
F, GR = f2.construir(c, TODO)
rng = np.random.default_rng(3)
dias_prueba = [pd.Timestamp(x) for x in ['2024-02-05', '2024-07-04', '2024-11-12', '2025-03-27', '2025-06-30']]
cambian = pd.Series(0, index=F.columns)
for d in dias_prueba:
    Fp, _ = f2.construir(replace(f2.perturbar(c, d, rng), dias=pd.DatetimeIndex([d])), TODO)
    cambian += (~np.isclose(F.loc[d].values.astype(float), Fp.loc[d].values.astype(float), equal_nan=True)).astype(int)
tab = pd.DataFrame({'columnas': GR.value_counts(), 'columnas que cambian con basura en el futuro': (cambian > 0).groupby(GR).sum()}).fillna(0).astype(int)
tab.index = [f2.GRUPOS[k][1][:70] for k in tab.index]
display(tab); print(f"{len(F.columns)} columnas revisadas en {len(dias_prueba)} días; columnas que cambian: {int((cambian > 0).sum())}")

,columnas,columnas que cambian con basura en el futuro
"MTR rezagos d−8, d−9, d−10, d−14 (24 h c/u)",96,0
"MTR mismo día de la semana d−21, d−28",48,0
"calendario base: dow, feriado, Fourier anual k=1,2",12,0
congestión: componente de congestión del MDA de Monclova y de los corr,61,0
"clima nacional como serie: índice ponderado por consumo, calor acumula",18,0
días análogos: qué hizo el MTR en los días pasados más parecidos,27,0
"varias escalas y largo alcance: medias exponenciales, colas, persisten",81,0
"la red en el espacio: dispersión del MDA entre nodos, error de la red",31,0
"complejidad y correlaciones: entropías, persistencia, coherencia entre",18,0
"complejidad y correlaciones del MTR: entropías, persistencia, coherenc",16,0


744 columnas revisadas en 5 días; columnas que cambian: 0


Las 744 columnas, repartidas en los 20 grupos de variables, no cambian en ninguno de los 5 días probados (que cubren entrenamiento, desarrollo y test).

Ninguna variable usa MTR posterior a *d−8*, MDA posterior a *d* ni datos de otras fuentes posteriores a su hora de publicación. Que los modelos respeten ese mismo retraso al entrenar y reentrenar se verifica por separado en las pruebas de `tests/`.

Las mejoras que se ven en los notebooks siguientes no vienen de información del futuro.